# Day 070 — Exercise 5: ImageGenerator Class

**What you'll build:** `ImageGenerator` — a class that binds `generate_fn` at construction time and exposes `generate`, `batch`, and `grid` methods.

**Why it matters:** A class makes the generation pipeline reusable across many calls. Bind the mock at construction time for tests, swap in a real pipeline for production — no changes to callers.

In [ ]:
import math
from PIL import Image

STYLE_TEMPLATES = {
    'cinematic': {
        'positive': 'cinematic lighting, film grain, dramatic shadows',
        'negative': 'flat lighting, cartoon, illustration',
    },
    'photorealistic': {
        'positive': 'photorealistic, 8k uhd, high detail, DSLR',
        'negative': 'painting, drawing, blur, watermark',
    },
    'watercolor': {
        'positive': 'watercolor painting, soft edges, artistic',
        'negative': 'photorealistic, sharp, digital art',
    },
    'anime': {
        'positive': 'anime style, cel shaded, vibrant colours',
        'negative': 'photorealistic, watercolor, oil painting',
    },
}

def build_prompt(subject, style='', quality_tags=None, negative_tags=None):
    parts = [subject]
    if style: parts.append(style)
    if quality_tags: parts.extend(quality_tags)
    positive = ', '.join(p.strip() for p in parts if p.strip())
    negative = ', '.join(t.strip() for t in (negative_tags or []) if t.strip())
    return {'positive': positive, 'negative': negative}

def apply_style_template(base_prompt, template_name, templates=None):
    tmpl_dict = templates if templates is not None else STYLE_TEMPLATES
    tmpl = tmpl_dict.get(template_name)
    if tmpl is None:
        raise ValueError(f'Unknown style template {template_name!r}. Available: {list(tmpl_dict.keys())}')
    pos = base_prompt + ', ' + tmpl['positive'] if base_prompt.strip() else tmpl['positive']
    return {'positive': pos, 'negative': tmpl['negative']}

def generate_image(prompt, negative='', generate_fn=None,
                   width=512, height=512, steps=20,
                   guidance_scale=7.5, seed=42):
    if generate_fn is not None:
        return generate_fn(prompt, negative=negative, width=width,
                           height=height, steps=steps,
                           guidance_scale=guidance_scale, seed=seed)
    from diffusers import StableDiffusionPipeline
    import torch
    pipe = StableDiffusionPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5',
        torch_dtype=torch.float32,
    )
    generator = torch.Generator().manual_seed(seed)
    result = pipe(prompt, negative_prompt=negative or None,
                  width=width, height=height,
                  num_inference_steps=steps,
                  guidance_scale=guidance_scale,
                  generator=generator)
    return result.images[0]

def generate_variations(base_prompt, n_variations, generate_fn=None, **kwargs):
    images = []
    for i in range(n_variations):
        seed = i * 1000 + 42
        img = generate_image(base_prompt, generate_fn=generate_fn, seed=seed, **kwargs)
        images.append(img)
    return images

def create_image_grid(images, cols=2):
    if not images:
        raise ValueError('images list must not be empty')
    cols = max(1, min(cols, len(images)))
    rows = math.ceil(len(images) / cols)
    w, h = images[0].size
    grid = Image.new('RGB', (cols * w, rows * h), 'white')
    for i, img in enumerate(images):
        col = i % cols
        row = i // cols
        if img.size != (w, h):
            img = img.resize((w, h), Image.Resampling.LANCZOS)
        grid.paste(img, (col * w, row * h))
    return grid

_mock_gen = lambda prompt, **kw: Image.new('RGB', (kw.get('width', 512), kw.get('height', 512)), 'steelblue')


## Task

Implement `ImageGenerator`:

**`__init__(self, model_id, generate_fn=None)`:** store both as instance attributes.

**`generate(self, prompt, negative='', width=512, height=512, steps=20, guidance_scale=7.5, seed=42)`:**
Delegate to `generate_image(prompt, ..., generate_fn=self._generate_fn, ...)`. Return the result.

**`batch(self, prompts, **kwargs)`:** `return [self.generate(p, **kwargs) for p in prompts]`

**`grid(self, prompts, cols=2, **kwargs)`:** `return create_image_grid(self.batch(prompts, **kwargs), cols=cols)`

## Your Implementation

In [ ]:
class ImageGenerator:
    """Generate images from text prompts using a diffusion model.

    Inject generate_fn for testing without GPU or diffusers installed::

        mock = lambda p, **kw: Image.new('RGB', (kw.get('width', 512), kw.get('height', 512)), 'steelblue')
        gen = ImageGenerator(generate_fn=mock)
    """

    def __init__(self, model_id: str = 'runwayml/stable-diffusion-v1-5',
                 generate_fn=None) -> None:
        raise NotImplementedError

    def generate(self, prompt: str, negative: str = '',
                 width: int = 512, height: int = 512,
                 steps: int = 20, guidance_scale: float = 7.5,
                 seed: int = 42):
        """Generate a single image from a prompt. Returns PIL.Image.Image."""
        raise NotImplementedError

    def batch(self, prompts: list, **kwargs) -> list:
        """Generate one image per prompt. Returns list[Image.Image]."""
        raise NotImplementedError

    def grid(self, prompts: list, cols: int = 2, **kwargs):
        """Generate images for all prompts and stitch into a grid."""
        raise NotImplementedError


In [ ]:
class ImageGenerator:
    def __init__(self, model_id='runwayml/stable-diffusion-v1-5',
                 generate_fn=None) -> None:
        self._model_id   = model_id
        self._generate_fn = generate_fn

    def generate(self, prompt, negative='', width=512, height=512,
                 steps=20, guidance_scale=7.5, seed=42):
        return generate_image(
            prompt, negative=negative,
            generate_fn=self._generate_fn,
            width=width, height=height,
            steps=steps, guidance_scale=guidance_scale, seed=seed,
        )

    def batch(self, prompts, **kwargs):
        return [self.generate(p, **kwargs) for p in prompts]

    def grid(self, prompts, cols=2, **kwargs):
        return create_image_grid(self.batch(prompts, **kwargs), cols=cols)


## Automated checks

In [ ]:
score, total = 0, 5
try:
    gen = ImageGenerator(generate_fn=_mock_gen)

    # generate returns PIL Image of correct size
    img = gen.generate('a sunset', width=128, height=64)
    assert isinstance(img, Image.Image) and img.size == (128, 64)
    score += 1; print("\u2705 generate returns PIL Image of correct size")

    # batch returns list of correct length
    prompts = ['a cat', 'a dog', 'a bird']
    imgs = gen.batch(prompts, width=32, height=32)
    assert len(imgs) == 3 and all(isinstance(i, Image.Image) for i in imgs)
    score += 1; print("\u2705 batch returns list of 3 PIL Images")

    # grid returns stitched image of correct dimensions
    grid = gen.grid(prompts, cols=3, width=32, height=32)
    assert grid.size == (96, 32), f"Expected (96,32), got {grid.size}"
    score += 1; print("\u2705 grid (3 prompts, 3 cols) returns (96,32) image")

    # 2-col grid: 3 prompts → 2 rows
    grid2 = gen.grid(prompts, cols=2, width=32, height=32)
    assert grid2.size == (64, 64), f"Expected (64,64), got {grid2.size}"
    score += 1; print("\u2705 grid (3 prompts, 2 cols) returns (64,64) image")

    # generate_fn is stored and forwarded
    captured = {}
    def _cap(prompt, **kw):
        captured['guidance_scale'] = kw.get('guidance_scale')
        return Image.new('RGB', (kw.get('width', 64), kw.get('height', 64)), 'red')
    gen2 = ImageGenerator(generate_fn=_cap)
    gen2.generate('test', guidance_scale=11.0)
    assert abs(captured.get('guidance_scale', 0) - 11.0) < 0.001
    score += 1; print("\u2705 generate_fn and params forwarded correctly")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class ImageGenerator:
    def __init__(self, model_id='runwayml/stable-diffusion-v1-5',
                 generate_fn=None) -> None:
        self._model_id   = model_id
        self._generate_fn = generate_fn

    def generate(self, prompt, negative='', width=512, height=512,
                 steps=20, guidance_scale=7.5, seed=42):
        return generate_image(
            prompt, negative=negative,
            generate_fn=self._generate_fn,
            width=width, height=height,
            steps=steps, guidance_scale=guidance_scale, seed=seed,
        )

    def batch(self, prompts, **kwargs):
        return [self.generate(p, **kwargs) for p in prompts]

    def grid(self, prompts, cols=2, **kwargs):
        return create_image_grid(self.batch(prompts, **kwargs), cols=cols)
```

**Why delegate `generate` to the module-level `generate_image`?** The class should not duplicate the conditional logic for real vs mock. By passing `self._generate_fn` to the module function, all the dispatch logic lives in one place. The class is a thin wrapper that holds state (the mock) and provides a convenient interface.

</details>